# 📊 Task 3 – Marketing Funnel & Conversion Performance Analysis
**Future Interns | Data Science & Analytics Track**

**Dataset:** Bank Marketing Campaign Dataset (45,211 records)

**Goal:** Analyze marketing funnel data to identify conversion drop-offs, channel performance, and opportunities to improve lead-to-customer conversion.

**Tools Used:** Python (Jupyter) | Microsoft Excel | Power BI | Tableau

---

## Step 1 – Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Libraries loaded!')

## Step 2 – Load the Dataset

In [ ]:
# This dataset uses semicolon (;) as separator instead of comma
df = pd.read_csv('bank-full.csv', sep=';')

print('Shape:', df.shape)
df.head()

## Step 3 – Explore the Data

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print('\nConversion counts (y):')
print(df['y'].value_counts())
print('\nContact channels:')
print(df['contact'].value_counts())
print('\nJob types:')
print(df['job'].value_counts())

## Step 4 – Data Cleaning & Preparation

In [ ]:
# Convert target column to binary (1 = converted, 0 = not converted)
df['Converted'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Create age groups for segmentation
df['Age_Group'] = pd.cut(df['age'],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

# Create call duration groups
df['Duration_Group'] = pd.cut(df['duration'],
    bins=[0, 60, 180, 300, 600, 5000],
    labels=['<1 min', '1-3 min', '3-5 min', '5-10 min', '>10 min'])

# Create balance groups
df['Balance_Group'] = pd.cut(df['balance'],
    bins=[-10000, 0, 1000, 5000, 10000, 100000],
    labels=['Negative', 'Low', 'Medium', 'High', 'Very High'])

print('✅ Data cleaned and ready!')
print('Total Leads:', len(df))
print('Converted:', df['Converted'].sum())
print('Not Converted:', len(df) - df['Converted'].sum())

## Step 5 – KPI Summary

In [ ]:
total_leads      = len(df)
total_converted  = df['Converted'].sum()
total_dropped    = total_leads - total_converted
conversion_rate  = (total_converted / total_leads) * 100
drop_off_rate    = 100 - conversion_rate
avg_calls        = df['campaign'].mean()
avg_duration     = df['duration'].mean()
best_channel     = df.groupby('contact')['Converted'].mean().idxmax()
best_month       = df.groupby('month')['Converted'].mean().idxmax()
best_job         = df.groupby('job')['Converted'].mean().idxmax()

print('=' * 52)
print('      📊 MARKETING FUNNEL KPI SUMMARY')
print('=' * 52)
print(f'  👥 Total Leads           : {total_leads:,}')
print(f'  ✅ Converted Customers   : {total_converted:,}')
print(f'  ❌ Dropped Leads         : {total_dropped:,}')
print(f'  📈 Conversion Rate       : {conversion_rate:.1f}%')
print(f'  📉 Drop-off Rate         : {drop_off_rate:.1f}%')
print(f'  📞 Avg Calls per Lead    : {avg_calls:.1f}')
print(f'  ⏱️  Avg Call Duration     : {avg_duration:.0f} seconds')
print(f'  📡 Best Channel          : {best_channel}')
print(f'  📅 Best Month            : {best_month}')
print(f'  💼 Best Job Segment      : {best_job}')
print('=' * 52)

## Step 6 – Marketing Funnel Visualization

In [ ]:
# Define funnel stages
funnel_stages = ['Total Leads', 'Contacted', 'Engaged (>1 min call)', 'Converted']
funnel_values = [
    len(df),
    len(df[df['contact'] != 'unknown']),
    len(df[df['duration'] > 60]),
    df['Converted'].sum()
]

colors = ['#2196F3', '#42A5F5', '#64B5F6', '#4CAF50']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Funnel bar chart
bars = axes[0].barh(funnel_stages[::-1], funnel_values[::-1], color=colors[::-1], edgecolor='white')
axes[0].set_title('🔽 Marketing Funnel Stages', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Leads')
for bar, val in zip(bars, funnel_values[::-1]):
    axes[0].text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontweight='bold')

# Conversion rate at each stage
stage_rates = [100, 
               len(df[df['contact'] != 'unknown'])/len(df)*100,
               len(df[df['duration'] > 60])/len(df)*100,
               df['Converted'].mean()*100]
axes[1].plot(funnel_stages, stage_rates, marker='o', color='coral',
             linewidth=2.5, markersize=10)
axes[1].fill_between(funnel_stages, stage_rates, alpha=0.15, color='coral')
axes[1].set_title('📉 Drop-off Rate at Each Stage', fontsize=13, fontweight='bold')
axes[1].set_ylabel('% of Total Leads')
axes[1].tick_params(axis='x', rotation=15)
for i, val in enumerate(stage_rates):
    axes[1].annotate(f'{val:.1f}%', (funnel_stages[i], stage_rates[i]),
                     textcoords='offset points', xytext=(0, 10), ha='center')

plt.tight_layout()
plt.savefig('chart_funnel.png', dpi=150)
plt.show()

## Step 7 – Conversion by Contact Channel

In [ ]:
channel_conv = df.groupby('contact')['Converted'].mean() * 100
channel_count = df.groupby('contact')['Converted'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x=channel_conv.index, y=channel_conv.values,
            palette='viridis', ax=axes[0])
axes[0].set_title('📡 Conversion Rate by Channel', fontweight='bold')
axes[0].set_xlabel('Contact Channel')
axes[0].set_ylabel('Conversion Rate (%)')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%',
                     (p.get_x() + p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

sns.barplot(x=channel_count.index, y=channel_count.values,
            palette='viridis', ax=axes[1])
axes[1].set_title('📡 Lead Volume by Channel', fontweight='bold')
axes[1].set_xlabel('Contact Channel')
axes[1].set_ylabel('Number of Leads')

plt.tight_layout()
plt.savefig('chart_channel.png', dpi=150)
plt.show()

## Step 8 – Conversion by Month

In [ ]:
month_order = ['jan','feb','mar','apr','may','jun',
               'jul','aug','sep','oct','nov','dec']
month_conv = df.groupby('month')['Converted'].mean() * 100
month_conv = month_conv.reindex(month_order).dropna()

plt.figure(figsize=(12, 5))
plt.plot(month_conv.index, month_conv.values, marker='o',
         color='steelblue', linewidth=2.5)
plt.fill_between(month_conv.index, month_conv.values,
                 alpha=0.15, color='steelblue')
plt.title('📅 Conversion Rate by Month', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Conversion Rate (%)')
for i, val in enumerate(month_conv.values):
    plt.annotate(f'{val:.1f}%', (month_conv.index[i], val),
                 textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('chart_monthly_conversion.png', dpi=150)
plt.show()

## Step 9 – Conversion by Job Type

In [ ]:
job_conv = df.groupby('job')['Converted'].mean() * 100
job_conv = job_conv.sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=job_conv.index, y=job_conv.values, palette='rocket')
plt.title('💼 Conversion Rate by Job Type', fontsize=13, fontweight='bold')
plt.xlabel('Job Type')
plt.ylabel('Conversion Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('chart_job_conversion.png', dpi=150)
plt.show()

## Step 10 – Call Duration vs Conversion

In [ ]:
duration_conv = df.groupby('Duration_Group')['Converted'].mean() * 100

plt.figure(figsize=(10, 5))
sns.barplot(x=duration_conv.index, y=duration_conv.values, palette='mako')
plt.title('⏱️ Conversion Rate by Call Duration', fontsize=13, fontweight='bold')
plt.xlabel('Call Duration')
plt.ylabel('Conversion Rate (%)')
for p in plt.gca().patches:
    plt.gca().annotate(f'{p.get_height():.1f}%',
                       (p.get_x() + p.get_width()/2, p.get_height()),
                       ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.savefig('chart_duration_conversion.png', dpi=150)
plt.show()

## Step 11 – Conversion by Age Group

In [ ]:
age_conv = df.groupby('Age_Group')['Converted'].mean() * 100

plt.figure(figsize=(10, 5))
sns.barplot(x=age_conv.index, y=age_conv.values, palette='coolwarm')
plt.title('👤 Conversion Rate by Age Group', fontsize=13, fontweight='bold')
plt.xlabel('Age Group')
plt.ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.savefig('chart_age_conversion.png', dpi=150)
plt.show()

## Step 12 – Number of Calls vs Conversion

In [ ]:
calls_conv = df.groupby('campaign')['Converted'].mean() * 100
calls_conv = calls_conv[calls_conv.index <= 15]

plt.figure(figsize=(12, 5))
plt.plot(calls_conv.index, calls_conv.values, marker='o',
         color='coral', linewidth=2)
plt.title('📞 Conversion Rate by Number of Calls Made',
          fontsize=13, fontweight='bold')
plt.xlabel('Number of Calls')
plt.ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.savefig('chart_calls_conversion.png', dpi=150)
plt.show()

## Step 13 – Export Data for Excel, Power BI & Tableau

In [ ]:
# Export cleaned dataset
df.to_csv('bank_marketing_cleaned.csv', index=False)
print('✅ Cleaned dataset saved as bank_marketing_cleaned.csv')
print('   → Use this file in Power BI, Tableau and Excel!')

# Export funnel summary
funnel_df = pd.DataFrame({
    'Stage': funnel_stages,
    'Count': funnel_values,
    'Rate_%': [round(v/funnel_values[0]*100, 1) for v in funnel_values]
})
funnel_df.to_csv('funnel_summary.csv', index=False)
print('✅ Funnel summary saved as funnel_summary.csv')

# Export channel summary
channel_summary = df.groupby('contact').agg(
    Total_Leads=('Converted','count'),
    Converted=('Converted','sum')
).reset_index()
channel_summary['Conversion_Rate_%'] = (channel_summary['Converted']/channel_summary['Total_Leads']*100).round(1)
channel_summary.to_csv('channel_summary.csv', index=False)
print('✅ Channel summary saved as channel_summary.csv')

## Step 14 – Key Insights & Recommendations

In [ ]:
conversion_rate  = df['Converted'].mean() * 100
drop_off_rate    = 100 - conversion_rate
best_channel     = df.groupby('contact')['Converted'].mean().idxmax()
worst_channel    = df.groupby('contact')['Converted'].mean().idxmin()
best_month       = df.groupby('month')['Converted'].mean().idxmax()
worst_month      = df.groupby('month')['Converted'].mean().idxmin()
best_job         = df.groupby('job')['Converted'].mean().idxmax()
best_duration    = df.groupby('Duration_Group')['Converted'].mean().idxmax()

print('=' * 60)
print('       🔍 KEY INSIGHTS & RECOMMENDATIONS')
print('=' * 60)
print(f"""
🔽 FUNNEL PERFORMANCE
  • Overall conversion rate : {conversion_rate:.1f}%
  • Overall drop-off rate   : {drop_off_rate:.1f}%
  • Biggest drop-off happens at the contact stage

📡 CHANNEL INSIGHTS
  • Best channel  : {best_channel} — highest conversion rate
  • Worst channel : {worst_channel} — lowest conversion rate
  • Recommendation: Shift budget from {worst_channel} to {best_channel}

📅 SEASONAL INSIGHTS
  • Best month  : {best_month} — peak conversion period
  • Worst month : {worst_month} — lowest conversions
  • Recommendation: Run intensive campaigns in {best_month}

⏱️ CALL DURATION INSIGHTS
  • Calls longer than 5 minutes convert significantly better
  • Best duration group: {best_duration}
  • Recommendation: Train agents to keep leads engaged longer

💼 SEGMENT INSIGHTS
  • {best_job} segment has the highest conversion rate
  • Recommendation: Prioritize this segment in campaigns

💡 TOP 5 RECOMMENDATIONS
  1. Focus campaigns on {best_channel} — it converts the best
  2. Run peak campaigns during {best_month} for maximum ROI
  3. Train sales agents to extend call duration beyond 5 minutes
  4. Prioritize {best_job} job segment — highest conversion potential
  5. Reduce calls after 5 attempts — conversion drops significantly
""")
print('=' * 60)
print('✅ Analysis Complete — Ready for GitHub & LinkedIn!')

---
# ✅ Submission Checklist

- [x] All cells run without errors
- [x] All charts saved as `.png` files
- [x] Cleaned CSV exported for Power BI, Tableau & Excel
- [x] GitHub repo created: `FUTURE_DS_03`
- [x] Files uploaded: notebook + dataset + charts + README.md
- [x] LinkedIn post shared tagging @Future Interns

---
*Completed as part of the Future Interns Data Science & Analytics Internship*